In [3]:
%pip install pennylane pennylane-lightning qiskit qiskit-aer matplotlib scipy pyqsp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pennylane as qml
import numpy as np

In [5]:
def controlled_projector_phase(phi, new_ancilla, old_ancillae):
    """
    Implements the e^{i phi} * CPi_{0^a} operator described in the algorithm.
    Applies a phase shift of 'phi' if the new_ancilla is in state |1> 
    AND all old_ancillae are in state |0>.
    """
    # Control condition: old_ancillae must all be in state |0>
    ctrl_vals = [0] * len(old_ancillae)
    
    # Apply the PhaseShift on the new_ancilla, 
    # controlled by the old_ancillae register being in the |0...0> state.
    qml.ctrl(qml.RX(phi, wires=new_ancilla), 
             control=old_ancillae, 
             control_values=ctrl_vals)

def base_block_encoding(A, old_ancillae, system_qubits):
    """
    Implements the underlying Block Encoding U for matrix A.
    """
    # PennyLane's BlockEncode automatically embeds A into a larger unitary.
    qml.BlockEncode(A, wires=old_ancillae + system_qubits)

def single_ancilla_phase_correction(A, phi_angles, new_ancilla, old_ancillae, system_qubits):
    """
    Core implementation of Algorithm 1: Single-Ancilla Phase Correction.
    
    Args:
        A (array): The matrix to be block encoded.
        phi_angles (array): Vector of d phase angles (phi_1, ..., phi_d).
        new_ancilla (list): Wire index for the single new ancilla qubit.
        old_ancillae (list): Wire indices for the original block encoding ancillae.
        system_qubits (list): Wire indices for the target system qubits.
    """
    d = len(phi_angles)
    # The loop runs (d-1)/2 times as per the summation index in the algorithm
    half_d = int((d - 1) / 2)
    
    # Step 1: Initialize with the base block encoding
    # U_tilde = I \otimes U
    base_block_encoding(A, old_ancillae, system_qubits)
    
    # Step 2: Alternating sequence of phase corrections and U / U_dagger
    # Note: Python uses 0-based indexing, so phi_angles[0] corresponds to phi_1.
    for j in range(1, half_d + 1):
        # Identify angles for the current iteration: phi_{2j-1} and phi_{2j}
        phi_odd = phi_angles[2*j - 2]   # Corresponding to phi_{2j-1}
        phi_even = phi_angles[2*j - 1]  # Corresponding to phi_{2j}
        
        # Apply (e^{i * phi_{2j-1}} * CPi_{0^a} \otimes I)
        controlled_projector_phase(phi_odd, new_ancilla, old_ancillae)
        
        # Apply (I \otimes U_dagger)
        qml.adjoint(base_block_encoding)(A, old_ancillae, system_qubits)
        
        # Apply (e^{i * phi_{2j}} * CPi_{0^a} \otimes I)
        controlled_projector_phase(phi_even, new_ancilla, old_ancillae)
        
        # Apply (I \otimes U)
        base_block_encoding(A, old_ancillae, system_qubits)
        
    # Step 3: Final phase adjustment
    # Apply e^{i * phi_d}
    phi_d = phi_angles[-1]
    qml.GlobalPhase(phi_d)


# ==========================================
# Example Execution and Testing
# ==========================================

# 1. Define Matrix A (Constraint: ||A|| <= 1)
A = np.array([[0.3, 0.0], [0.0, 0.5]]) 
n_system = 1 
a_old = 1    

# 2. Define Register Wires
new_ancilla = [0]
old_ancillae = [1]
system_qubits = [2]

# 3. Provide Phase Angles
# In practice, these angles are pre-computed using QSP/QSVT angle finding techniques.
# Here we use dummy angles for demonstration.
d = 3 # d must be odd
phi_angles = np.array([0.4, 0.5, 0.2]) 

# 4. Create and Run the Quantum Node (QNode)
dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def circuit():
    # Prepare an initial state for the system
    qml.Hadamard(wires=system_qubits[0])
    
    # Execute the algorithm
    single_ancilla_phase_correction(A, phi_angles, new_ancilla, old_ancillae, system_qubits)
    
    return qml.state()

# Output the circuit visualization and the final state
print("Quantum Circuit Visualization:")
print(qml.draw(circuit)())
print("\nFinal State Vector:")
print(circuit())

Quantum Circuit Visualization:
0: ─────────────────────╭RX(0.40)───────────────────╭RX(0.50)───────────────── ···
1: ────╭BlockEncode(M0)─╰○────────╭BlockEncode(M0)†─╰○────────╭BlockEncode(M0) ···
2: ──H─╰BlockEncode(M0)───────────╰BlockEncode(M0)†───────────╰BlockEncode(M0) ···

0: ··· ─╭GlobalPhase(0.20)─┤  State
1: ··· ─├GlobalPhase(0.20)─┤  State
2: ··· ─╰GlobalPhase(0.20)─┤  State

M0 = 
[[0.3 0. ]
 [0.  0.5]]

Final State Vector:
[ 0.19638798-0.03980981j  0.32462264-0.06580427j  0.63765177-0.12925841j
  0.57422626-0.11640143j -0.01875724-0.09253237j -0.03113723-0.15360478j
 -0.03302058-0.16289565j -0.02976123-0.14681676j]


In [6]:
# 1. Define Matrix A and Angles
A = np.array([[0.8, 0.1], [0.1, 0.8]]) # A non-trivial matrix
phi_angles = np.array([0.1, 0.1, 0.4]) # Dummy angles for demo

# 2. Define Wires
new_ancilla = [0]
old_ancillae = [1]
system_qubits = [2]

# 3. Create Device
dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def circuit_with_postselection():
    # Initial state: System is in |+> state
    qml.Hadamard(wires=system_qubits[0])
    
    # Apply the Phase Correction Algorithm
    single_ancilla_phase_correction(A, phi_angles, new_ancilla, old_ancillae, system_qubits)
    
    # POST-SELECTION:
    # We measure the ancillae and force the outcome to be 0.
    # PennyLane will return the state of the system qubits conditioned on this result.
    qml.measure(new_ancilla[0], postselect=0)
    for w in old_ancillae:
        qml.measure(w, postselect=0)
        
    # Return the probabilities of the system qubit
    return qml.probs(wires=system_qubits)

# --- Execution ---

print("Executing circuit with post-selection...")
results = circuit_with_postselection()

print("\n--- Results ---")
print(f"Probabilities of System Qubit (State |0> vs |1>):")
print(f"|0>: {results[0]:.4f}")
print(f"|1>: {results[1]:.4f}")

# Verification: Compare with A |+>
initial_psi = np.array([1, 1]) / np.sqrt(2)
expected_unnormalized = A @ initial_psi
expected_probs = np.abs(expected_unnormalized)**2
expected_probs /= np.sum(expected_probs) # Renormalize for comparison

print("\n--- Theoretical Expected Probs (if angles were identity) ---")
print(f"|0>: {expected_probs[0]:.4f}")
print(f"|1>: {expected_probs[1]:.4f}")

Executing circuit with post-selection...

--- Results ---
Probabilities of System Qubit (State |0> vs |1>):
|0>: 0.5000
|1>: 0.5000

--- Theoretical Expected Probs (if angles were identity) ---
|0>: 0.5000
|1>: 0.5000


In [7]:
A = np.array([[0.8, 0.2], 
              [0.2, 0.1]])
new_ancilla = [0]
old_ancillae = [1]
system_qubits = [2]
dev = qml.device("default.qubit", wires=3)

def run_test(angles, use_hadamard=True):
    @qml.qnode(dev)
    def circuit():
        if use_hadamard:
            qml.Hadamard(wires=new_ancilla[0])
        qml.Hadamard(wires=system_qubits[0])
        single_ancilla_phase_correction(A, angles, new_ancilla, old_ancillae, system_qubits)
        return qml.probs(wires=new_ancilla + old_ancillae + system_qubits)
    
    probs = circuit()
    return np.sum(probs[0:2])

print(f"{'Test Case':<30} | {'Angles':<20} | {'Success Prob':<12}")
print("-" * 70)

p1 = run_test(np.array([0.0, 0.0, 0.0]), use_hadamard=True)
print(f"{'1. Zero Angles (Baseline)':<30} | {'[0, 0, 0]':<20} | {p1:.6f}")

p2 = run_test(np.array([np.pi, np.pi, np.pi]), use_hadamard=True)
print(f"{'2. Destructive (Pi)':<30} | {'[PI, PI, PI]':<20} | {p2:.6f}")

p3_static = run_test(np.array([0.5, 1.2, 0.8]), use_hadamard=False)
p3_dynamic = run_test(np.array([0.5, 1.2, 0.8]), use_hadamard=True)
print(f"{'3a. Static (No Hadamard)':<30} | {'[0.5, 1.2, 0.8]':<20} | {p3_static:.6f}")
print(f"{'3b. Dynamic (With Hadamard)':<30} | {'[0.5, 1.2, 0.8]':<20} | {p3_dynamic:.6f}")

p4 = run_test(np.array([0.1, 0.5, 0.1]), use_hadamard=True)
print(f"{'4. ':<30} | { '[0.1, 0.5, 0.1]':<20} | {p4:.6f}")

Test Case                      | Angles               | Success Prob
----------------------------------------------------------------------
1. Zero Angles (Baseline)      | [0, 0, 0]            | 0.272500
2. Destructive (Pi)            | [PI, PI, PI]         | 0.354281
3a. Static (No Hadamard)       | [0.5, 1.2, 0.8]      | 0.264489
3b. Dynamic (With Hadamard)    | [0.5, 1.2, 0.8]      | 0.292916
4.                             | [0.1, 0.5, 0.1]      | 0.274337
